# Unión de las bases geoespaciales — Malla Vial, Puentes, Malla Peatonal, Semáforos con los datos de la siniestralidad.

Este cuaderno **importa directamente las 4 capas ya procesadas y unidas** del cuaderno **Union_bases_malla_vial**:

- `Union_bases_malla_vial` → exporta `Data/processed/malla_urbana_unificada` 

> **Importante:** este notebook ya solo trabaja con esa union establecida

**Enfoque de unión:**

1. Se toma la **malla vial unificada** (salida de `Union_bases_malla_vial`) como capa base.
2. Cada **siniestro** se relaciona con el segmento vial más cercano mediante `sjoin_nearest`, igual que se hizo con puentes, semáforos y malla peatonal en el cuaderno de unión.

El cuaderno está organizado así:

1. Librerías
2. Malla vial unificada (capa base)
3. Datos de siniestralidad
4. Unión espacial de siniestros con la malla vial

## 1. Librerías

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
plt.rcParams['figure.figsize'] = (9, 6)

## 2. Malla vial unificada (capa base)

In [5]:
malla = gpd.read_file('Data/processed/malla_urbana_unificada.gpkg',layer="malla_vial")

### 2.1 Vista previa de los datos

In [6]:
malla

,vial_id,MVITIPO,MVISVIA,longitud_m,geometry
0,96282,KR,SD,35.425891,"MULTILINESTRING ((997702.139 993255.081, 99767..."
1,62558,CL,B,25.743885,"MULTILINESTRING ((989012.577 1005585.74, 98899..."
2,33964,TV,B,95.373241,"MULTILINESTRING ((1006180.996 1007998.51, 1006..."
3,29147,CL,B,117.076476,"MULTILINESTRING ((1005380.357 1018221.924, 100..."
4,133742,KR,B,59.005266,"MULTILINESTRING ((992300.356 1005275.881, 9923..."
...,...,...,...,...,...
136952,83672,CL,B,35.415273,"MULTILINESTRING ((996031.881 1016116.935, 9960..."
136953,118082,CL,SD,23.209613,"MULTILINESTRING ((996509.218 993242.225, 99652..."
136954,36662,CL,B,41.861484,"MULTILINESTRING ((1000720.865 998938.497, 1000..."
136955,58906,KR,SD,90.254566,"MULTILINESTRING ((996262.641 989612.472, 99621..."


## 3. Datos de siniestralidad

In [4]:
siniestros = gpd.read_file("Data/processed/siniestros_procesados.gpkg")

if siniestros.crs.to_epsg() != 3116:
    siniestros = siniestros.to_crs(epsg=3116)

print(f"Registros de siniestros: {len(siniestros)}")
print(siniestros.crs)

Registros de siniestros: 199146
EPSG:3116


## 4. Unión espacial de siniestros con la malla vial

In [8]:
# Mismo patrón que puentes/semáforos/peatonal: sjoin_nearest asigna un único vial_id
# por siniestro (el más cercano), evitando que un evento cerca de una intersección
# se cuente para más de un segmento.
UMBRAL_SINIESTROS_M = 30  # más estricto que el 100m de las otras capas: aquí define el target, no contexto

siniestros_asignados = gpd.sjoin_nearest(
    siniestros, malla[["vial_id", "geometry"]],
    max_distance=UMBRAL_SINIESTROS_M, distance_col="dist_m"
)

# Reporte de cobertura, igual que en las capas anteriores — necesario para la sección
# de depuración/auditoría de tu tesis (antes/después por capa).
sin_match = siniestros["geometry"].size - siniestros_asignados["dist_m"].notna().sum()
print(f"Siniestros sin segmento asignado a {UMBRAL_SINIESTROS_M}m: {sin_match}")

siniestros_asignados = siniestros_asignados[siniestros_asignados["dist_m"].notna()].copy()

Siniestros sin segmento asignado a 30m: 1060
